In [ ]:
import os
import time
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
import torch
import torch.nn as nn

In [ ]:
COMPETITION_DIR      = '/kaggle/input/competitions/birdclef-2026'
CHECKPOINT_PATH      = '/kaggle/input/datasets/joshgreen/birdclef-2026-checkpoint/checkpoint.pt'
PERCH_MODEL_PATH     = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8'
TAXONOMY_PATH        = os.path.join(COMPETITION_DIR, 'taxonomy.csv')
TRAIN_SOUNDSCAPE_DIR = os.path.join(COMPETITION_DIR, 'train_soundscapes')

SAMPLE_RATE    = 32000
CHUNK_SECONDS  = 5
CHUNK_SAMPLES  = SAMPLE_RATE * CHUNK_SECONDS
EMBEDDING_DIM  = 1280
HIDDEN         = 512
PERCH_BATCH    = 256
MLP_BATCH      = 1024

taxonomy = pd.read_csv(TAXONOMY_PATH)
class_labels = sorted(taxonomy['primary_label'].astype(str).tolist())
n_classes = len(class_labels)
print(f'{n_classes} classes')

In [ ]:
tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)

perch = tf.saved_model.load(PERCH_MODEL_PATH)
infer = tf.function(perch.infer_tf)
_ = infer(tf.zeros([1, CHUNK_SAMPLES], dtype=tf.float32))  # warm up
print('Perch model loaded and warmed up')

In [ ]:
model = nn.Sequential(
    nn.Linear(EMBEDDING_DIM, HIDDEN),
    nn.ReLU(),
    nn.Dropout(0.0),
    nn.Linear(HIDDEN, n_classes),
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()
print(f'Checkpoint loaded (val_step={ckpt["val_step"]})')

In [ ]:
# Pass 1: collect all chunks
soundscapes = sorted(f for f in os.listdir(TRAIN_SOUNDSCAPE_DIR) if f.endswith('.ogg'))
print(f'{len(soundscapes)} train soundscapes')

all_chunks  = []
all_row_ids = []

t0 = time.perf_counter()
for idx, filename in enumerate(soundscapes):
    sig, rate = sf.read(os.path.join(TRAIN_SOUNDSCAPE_DIR, filename), dtype='float32')
    soundscape_id = filename.split('.')[0]

    for i in range(0, len(sig), CHUNK_SAMPLES):
        chunk = sig[i : i + CHUNK_SAMPLES]
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))
        all_chunks.append(chunk)
        end_time = (i // CHUNK_SAMPLES + 1) * CHUNK_SECONDS
        all_row_ids.append(f'{soundscape_id}_{end_time}')

    elapsed = time.perf_counter() - t0
    print(f'  [{idx+1}/{len(soundscapes)}] {filename}: {len(sig)/rate:.0f}s audio — {len(all_chunks)} chunks total — {elapsed:.1f}s elapsed')

print(f'\nPass 1 done: {len(all_chunks)} chunks in {time.perf_counter()-t0:.1f}s')

In [ ]:
# Pass 2: Perch embeddings
n = len(all_chunks)
embeddings = np.zeros((n, EMBEDDING_DIM), dtype=np.float32)

t0 = time.perf_counter()
for i in range(0, n, PERCH_BATCH):
    batch = np.stack(all_chunks[i : i + PERCH_BATCH])
    outputs = infer(tf.constant(batch))
    embeddings[i : i + len(batch)] = outputs['embedding'].numpy()

    done = min(i + PERCH_BATCH, n)
    elapsed = time.perf_counter() - t0
    rate = done / elapsed
    eta = (n - done) / rate if rate > 0 else 0
    print(f'  {done}/{n} chunks embedded — {rate:.1f} chunks/s — ETA {eta:.0f}s')

print(f'\nPass 2 done: {n} embeddings in {time.perf_counter()-t0:.1f}s')

In [ ]:
# Pass 3: MLP predictions
all_scores = np.zeros((n, n_classes), dtype=np.float32)

t0 = time.perf_counter()
with torch.no_grad():
    for i in range(0, n, MLP_BATCH):
        batch = torch.from_numpy(embeddings[i : i + MLP_BATCH]).to(device)
        scores = torch.sigmoid(model(batch)).cpu().numpy()
        all_scores[i : i + len(batch)] = scores

print(f'Pass 3 done: {n} predictions in {time.perf_counter()-t0:.1f}s')

In [ ]:
submission = pd.DataFrame(all_scores, columns=class_labels)
submission.insert(0, 'row_id', all_row_ids)
submission.to_csv('train_soundscape_preds.csv', index=False)
print(f'train_soundscape_preds.csv: {len(submission)} rows x {len(submission.columns)} columns')
submission.head()